# Extract Custom Fields from Your File

This notebook demonstrates how to use analyzers to extract custom fields from your input files.

## Prerequisites
1. Ensure Azure AI service is configured following [steps](../README.md#configure-azure-ai-service-resource)
2. Install the required packages to run the sample.

In [ ]:
%pip install -r ../requirements.txt

## Analyzer Templates

Below is a collection of analyzer templates designed to extract fields from various input file types.

These templates are highly customizable, allowing you to modify them to suit your specific needs. For additional verified templates from Microsoft, please visit [here](../analyzer_templates/README.md).

In [1]:
extraction_templates = {
    "invoice":            ('../analyzer_templates/invoice.json',         '../data/invoice.pdf'            ),
    "chart":              ('../analyzer_templates/image_chart.json',     '../data/pieChart.jpg'           ),
    "call_recording":     ('../analyzer_templates/call_recording_analytics.json', '../data/callCenterRecording.mp3'),
    "conversation_audio": ('../analyzer_templates/conversational_audio_analytics.json', '../data/callCenterRecording.mp3'),
    "marketing_video":    ('../analyzer_templates/marketing_video.json', '../data/FlightSimulator.mp4'              )
}

Specify the analyzer template you want to use and provide a name for the analyzer to be created based on the template.

In [4]:
import uuid

ANALYZER_TEMPLATE = "invoice"
ANALYZER_ID = "field-extraction-sample-" + str(uuid.uuid4())

(analyzer_template_path, analyzer_sample_file_path) = extraction_templates[ANALYZER_TEMPLATE]

## Create Azure AI Content Understanding Client

> The [AzureContentUnderstandingClient](../python/content_understanding_client.py) is a utility class containing functions to interact with the Content Understanding API. Before the official release of the Content Understanding SDK, it can be regarded as a lightweight SDK.


In [7]:
import logging
import json
import os
import sys
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

load_dotenv(find_dotenv())
logging.basicConfig(level=logging.INFO)

AZURE_AI_ENDPOINT = os.getenv("AZURE_AI_ENDPOINT")
AZURE_AI_API_VERSION = os.getenv("AZURE_AI_API_VERSION", "2025-05-01-preview")

# Add the parent directory to the path to use shared modules
parent_dir = Path(Path.cwd()).parent
sys.path.append(str(parent_dir))
from python.content_understanding_client import AzureContentUnderstandingClient

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

client = AzureContentUnderstandingClient(
    endpoint=AZURE_AI_ENDPOINT,
    api_version=AZURE_AI_API_VERSION,
    token_provider=token_provider,
    # x_ms_useragent="azure-ai-content-understanding-python/field_extraction", # This header is used for sample usage telemetry, please comment out this line if you want to opt out.
)

INFO:azure.identity._credentials.environment:No environment configuration found.
INFO:azure.identity._credentials.managed_identity:ManagedIdentityCredential will use IMDS
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'http://169.254.169.254/metadata/identity/oauth2/token?api-version=REDACTED&resource=REDACTED'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.23.0 Python/3.11.12 (Linux-5.15.146.1-microsoft-standard-WSL2-x86_64-with-glibc2.36)'
No body was attached to the request
INFO:azure.core.pipeline.policies.http_logging_policy:Response status: 400
Response headers:
    'Content-Type': 'application/json; charset=utf-8'
    'Server': 'IMDS/150.870.65.1544'
    'x-ms-request-id': '15cd9350-cbea-4979-ab1d-631d08db949d'
    'Date': 'Thu, 22 May 2025 06:31:21 GMT'
    'Content-Length': '88'
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'http://169.254.169.254/metadata/identity/oauth2/token?api-version=REDACTED&res

## Create Analyzer from the Template

In [8]:
response = client.begin_create_analyzer(ANALYZER_ID, analyzer_template_path=analyzer_template_path)
result = client.poll_result(response)

print(json.dumps(result, indent=2))

{
    "description": "Sample product demo video analyzer",
    "baseAnalyzerId": "prebuilt-videoAnalyzer",
    "config": {
        "locales": [
            "en-US",
            "fr-FR"
        ],
        "returnDetails": true,
        "enableFace": false,
        "disableFaceBlurring": false,
        "personDirectoryId": null,
        "segmentationMode": "auto",
        "disableContentFiltering": false
    },
    "fieldSchema": {
        "fields": {
            "Segments": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "SegmentId": {
                            "type": "string"
                        },
                        "Description": {
                            "type": "string",
                            "method": "generate",
                            "description": "Detailed summary of the video segment, focusing on product characteristics, lighting, and colo

ERROR:python.content_understanding_client:Error creating analyzer field-extraction-sample-36fd8d35-92d4-47f4-a4ed-8b37e8908db4: 400 Client Error: BadRequest for url: https://ai-foundry-sample-west-us.services.ai.azure.com/contentunderstanding/analyzers/field-extraction-sample-36fd8d35-92d4-47f4-a4ed-8b37e8908db4?api-version=2025-05-01-preview


{'error': {'code': 'API operation not supported for token authentication', 'message': 'ApiId multimodalintelligence-2025-05-01-preview OperationId ContentAnalyzers_CreateOrReplace not supported for CheckAccess.'}}


HTTPError: 400 Client Error: BadRequest for url: https://ai-foundry-sample-west-us.services.ai.azure.com/contentunderstanding/analyzers/field-extraction-sample-36fd8d35-92d4-47f4-a4ed-8b37e8908db4?api-version=2025-05-01-preview

## Extract Fields Using the Analyzer

After the analyzer is successfully created, we can use it to analyze our input files.

In [ ]:
response = client.begin_analyze(ANALYZER_ID, file_location=analyzer_sample_file_path)
result = client.poll_result(response)

print(json.dumps(result, indent=2))

## Clean Up
Optionally, delete the sample analyzer from your resource. In typical usage scenarios, you would analyze multiple files using the same analyzer.

In [ ]:
client.delete_analyzer(ANALYZER_ID)